# Tugas 3 | Preprocessing

Tugas 3 ini tujuannya untuk Preprocessing:

Dataset: detik_berita_2024.csv & pta_mulitT_full.csv

Tahapan: hapus stopwords -> hapus simbol -> spellcheck
-> stemming -> tokenisasi -> simpan hasil ke CSV

### Install library yang digunakan

In [ ]:
import sys
!{sys.executable} -m pip install sastrawi
!{sys.executable} -m pip install pyspellchecker 
!{sys.executable} -m pip install regex
!{sys.executable} -m pip install nltk

import pandas as pd
import re
from Sastrawi.Stemmer.StemmerFactory import StemmerFactory
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from spellchecker import SpellChecker
import nltk


[notice] A new release of pip is available: 25.1.1 -> 25.2
[notice] To update, run: python.exe -m pip install --upgrade pip



[notice] A new release of pip is available: 25.1.1 -> 25.2
[notice] To update, run: python.exe -m pip install --upgrade pip



[notice] A new release of pip is available: 25.1.1 -> 25.2
[notice] To update, run: python.exe -m pip install --upgrade pip



[notice] A new release of pip is available: 25.1.1 -> 25.2
[notice] To update, run: python.exe -m pip install --upgrade pip


### Load Dataset

- detik_berita_2024.csv  -> berisi berita hasil crawling dari portal Detik.com
- pta_mulitT_full.csv    -> berisi abstrak skripsi dari repository PTA Trunojoyo

In [17]:
detik = pd.read_csv("detik_berita_2024.csv")
pta   = pd.read_csv("pta_mulitT_full.csv")

print("Detik shape:", detik.shape)
print("PTA shape:", pta.shape)

print(detik.head())
print(pta.head())

Detik shape: (100, 5)
PTA shape: (2286, 5)
                                        Judul Berita  \
0  Marcus Gideon ke Ganda Pratama : Enggak Bisa S...   
1  Megawati Tampil Gemilang, Red Sparks Gebuk IBK...   
2  Punya Tim & Semangat Baru, Jakarta Pertamina E...   
3  Jakarta Pertamina Enduro Tak Mau Lagi Cuma Gas...   
4  Ducati ke Honda: Bangun Motor Butuh Waktu, Tak...   

                                                Link  Kategori     Tanggal  \
0  https://sport.detik.com/raket/d-7712417/marcus...  Olahraga  2024-12-31   
1  https://sport.detik.com/sport-lain/d-7712305/m...  Olahraga  2024-12-31   
2  https://sport.detik.com/sport-lain/d-7712206/p...  Olahraga  2024-12-31   
3  https://sport.detik.com/sport-lain/d-7712029/j...  Olahraga  2024-12-31   
4  https://sport.detik.com/moto-gp/d-7711970/duca...  Olahraga  2024-12-31   

                                          Isi Berita  
0  Marcus Fernaldi Gideonhadir di PelatnasPBSI. K...  
1  Megawati Hangestri PertiwimembawaRed S

### Stopword

Menghapus *stopwords* (kata-kata umum seperti "yang", "dan", "di") karena biasanya tidak menambah makna signifikan.

In [18]:
nltk.download('stopwords')
nltk.download('punkt')
nltk.download('punkt_tab')
stop_words = set(stopwords.words('indonesian'))

def remove_stopwords(text):
    tokens = word_tokenize(str(text))
    filtered = [w for w in tokens if not w.lower() in stop_words]
    return " ".join(filtered)

detik["clean_no_stopword"] = detik["Isi Berita"].apply(remove_stopwords)
pta["clean_no_stopword"]   = pta["abstrak"].apply(remove_stopwords)

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\endyzan\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\endyzan\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\endyzan\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


### Menghilangkan Symbol/Tanda Baca

Menghapus simbol/tanda baca agar teks hanya berisi huruf dan angka.

In [19]:
def remove_symbols(text):
    return re.sub(r'[^A-Za-z0-9\s]', '', str(text))

detik["clean_no_symbol"] = detik["clean_no_stopword"].apply(remove_symbols)
pta["clean_no_symbol"]   = pta["clean_no_stopword"].apply(remove_symbols)

### Cek Ejaan & Pembukaan Kata

Melakukan *spell checking* untuk memperbaiki kesalahan ejaan
dengan bantuan kamus bahasa Indonesia.

In [21]:
from spellchecker import SpellChecker
from tqdm import tqdm

# aktifkan tqdm untuk pandas
tqdm.pandas()

spell = SpellChecker(language=None, case_sensitive=False)
spell.word_frequency.load_text_file("./kamus_indonesia.txt") 

cache = {}

def correct_spelling(text):
    corrected = []
    for word in str(text).split():
        if word in cache:
            correction = cache[word]
        else:
            correction = spell.correction(word)
            cache[word] = correction if correction else word
        corrected.append(cache[word])
    return " ".join(corrected)

# gunakan progress_apply supaya ada progress bar
detik["clean_spellcheck"] = detik["clean_no_symbol"].progress_apply(correct_spelling)
pta["clean_spellcheck"]   = pta["clean_no_symbol"].progress_apply(correct_spelling)


100%|██████████| 2286/2286 [2:02:37<00:00,  3.22s/it]  


### Stemming

Melakukan *stemming* untuk mengubah kata ke bentuk dasarnya, 
misalnya "berlari" -> "lari", "mengajar" -> "ajar".

In [22]:
factory = StemmerFactory()
stemmer = factory.create_stemmer()

def stemming_text(text):
    return stemmer.stem(str(text))

detik["clean_stemmed"] = detik["clean_spellcheck"].apply(stemming_text)
pta["clean_stemmed"]   = pta["clean_spellcheck"].apply(stemming_text)


### Tokenisasi

Melakukan *tokenisasi* agar teks dipecah menjadi daftar kata per dokumen

In [23]:
def tokenize(text):
    return word_tokenize(str(text))

detik["tokenized"] = detik["clean_stemmed"].apply(tokenize)
pta["tokenized"]   = pta["clean_stemmed"].apply(tokenize)

### Saving Procesing

Hasil preprocessing disimpan dalam file CSV baru:

- detik_cleaned.csv
- pta_cleaned.csv

In [24]:
detik.to_csv("detik_cleaned.csv", index=False)
pta.to_csv("pta_cleaned.csv", index=False)

print("Preprocessing selesai! Hasil disimpan di detik_cleaned.csv dan pta_cleaned.csv")

Preprocessing selesai! Hasil disimpan di detik_cleaned.csv dan pta_cleaned.csv
